# Curvature of In-Context Learning — Kaggle T4 Runner

This notebook orchestrates all phases on the Kaggle T4 GPU.  
**Do not run phases you haven't scaffolded locally first.**

## Setup instructions (fresh Kaggle session)
1. Enable GPU (T4 x1) in Settings → Accelerator.
2. Add the GitHub repo as a Utility Script dataset OR run Cell 1 to clone.
3. If loading a saved checkpoint, add the output Dataset from a prior run.
4. Run cells in order. Each phase cell is guarded by an `assert` on prior results.

In [ ]:
# Cell 1: Environment setup
import subprocess, sys, os

REPO_URL = 'https://github.com/YOUR_USERNAME/curvature-icl.git'  # UPDATE THIS
REPO_DIR = '/kaggle/working/curvature-icl'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 2: Config + paths
from src.utils import load_config, ensure_dir

cfg = load_config('configs/default.yaml')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Override output paths to Kaggle working dir so they persist
cfg['paths']['checkpoint_dir'] = '/kaggle/working/results/checkpoints'
cfg['paths']['metrics_dir']    = '/kaggle/working/results/metrics'
cfg['paths']['figures_dir']    = '/kaggle/working/results/figures'

for p in cfg['paths'].values():
    ensure_dir(p)

print('Config loaded. Device:', DEVICE)
print('Context len:', cfg['task']['context_len'],
      'Input dim:', cfg['task']['input_dim'],
      'Model d_model:', cfg['model']['d_model'])

In [ ]:
# Cell 3: Phase 1 — Train toy ICL transformer
# Expected runtime: ~30–60 min on T4 for 100k steps
# To resume: set SKIP_TRAIN = True and ensure a checkpoint exists

SKIP_TRAIN = False  # set True to load existing checkpoint

from src.train_toy import train
from src.toy_icl import build_model
from src.utils import load_latest_checkpoint

if SKIP_TRAIN:
    model = build_model(cfg, DEVICE)
    step = load_latest_checkpoint(cfg['paths']['checkpoint_dir'], model, device=DEVICE)
    assert step > 0, 'No checkpoint found!'
    print(f'Loaded checkpoint at step {step}')
else:
    model = train(cfg=cfg, device=DEVICE)

model.eval()
print('Model ready.')

In [ ]:
# Cell 4: Diagnostic plots (soft gate — must check before proceeding)
import numpy as np
from scipy.stats import pearsonr
from src.toy_icl import sample_tasks, build_token_sequence, ols_predict
from src.utils import set_seed
from src.viz import plot_pred_vs_true

set_seed(cfg['seed'])
rng = np.random.default_rng(cfg['seed'])
task_cfg = cfg['task']

batch = sample_tasks(256, task_cfg['context_len'], task_cfg['input_dim'],
                      task_cfg['noise_std'], rng, DEVICE)

with torch.no_grad():
    seq = build_token_sequence(batch.xs, batch.ys, batch.x_query)
    preds = model(seq).cpu().numpy()

targets  = batch.y_query.cpu().numpy()
ols_preds = ols_predict(batch.xs, batch.ys, batch.x_query).cpu().numpy()

fig_path = plot_pred_vs_true(preds, targets, ols_preds,
                               fig_dir=cfg['paths']['figures_dir'])

ols_corr, _ = pearsonr(preds, ols_preds)
print(f'Model–OLS Pearson correlation: {ols_corr:.4f}')
assert ols_corr > 0.8, (
    f'SOFT GATE FAILED: OLS corr={ols_corr:.3f} < 0.8. '
    'ICL has not formed. Train longer or check model.'
)
print('Soft gate PASSED: ICL has formed.')

In [ ]:
# Cell 5: All understanding + diagnostic plots
from src.importance import score_all_methods
from src.curvature import curvature_spectrum, model_gram_spectrum
from src.eval_importance import sample_adversarial_tasks
from src.importance import loo_importance, firstorder_importance
from src.curvature import analytic_leverage
from src.viz import (
    plot_score_distributions, plot_score_correlation_heatmap,
    plot_curvature_spectrum, plot_leverage_vs_context_index,
    plot_readout_vs_analytic_corr, plot_adversarial_example,
)

scores = score_all_methods(model, batch, DEVICE)
scores_np = {k: v.cpu().numpy() for k, v in scores.items()}

plot_score_distributions(scores_np, fig_dir=cfg['paths']['figures_dir'])
plot_score_correlation_heatmap(scores_np, fig_dir=cfg['paths']['figures_dir'])

eigs_data  = curvature_spectrum(batch).cpu().numpy()
eigs_model = model_gram_spectrum(model, batch, DEVICE).cpu().numpy()
plot_curvature_spectrum(eigs_data, eigs_model, fig_dir=cfg['paths']['figures_dir'])

plot_leverage_vs_context_index(
    scores_np['curvature_analytic'], scores_np['curvature_readout'],
    fig_dir=cfg['paths']['figures_dir'])

plot_readout_vs_analytic_corr(
    scores_np['curvature_readout'], scores_np['curvature_analytic'],
    fig_dir=cfg['paths']['figures_dir'])

rng_adv = np.random.default_rng(cfg['seed'] + 5)
adv_batch = sample_adversarial_tasks(
    1, task_cfg['context_len'], task_cfg['input_dim'], task_cfg['noise_std'],
    task_cfg['adv_collinearity'], rng_adv, DEVICE)
adv_loo = loo_importance(model, adv_batch, DEVICE).cpu().numpy()[0]
adv_fo  = firstorder_importance(model, adv_batch, DEVICE).cpu().numpy()[0]
adv_lev = analytic_leverage(adv_batch).cpu().numpy()[0]
plot_adversarial_example(adv_lev, adv_fo, adv_loo,
                          fig_dir=cfg['paths']['figures_dir'])

print('All understanding/diagnostic figures done.')

In [ ]:
# Cell 6: Kill-test evaluation (Phase 1 GATE)
# Expected runtime: ~20–40 min on T4 (1000 random + 500 adversarial tasks)
from src.eval_importance import run_killtest
from src.utils import save_metrics
from src.viz import plot_rank_agreement, plot_topk_recall

verdict = run_killtest(model, cfg, device=DEVICE)

m_path = save_metrics(verdict, 'killtest_phase1', cfg['paths']['metrics_dir'])
plot_rank_agreement(verdict['summary'], fig_dir=cfg['paths']['figures_dir'])
plot_topk_recall(verdict['summary'],    fig_dir=cfg['paths']['figures_dir'])

gate = verdict['gate']
gap  = verdict['adv_spearman_gap_curvature_vs_firstorder']
print(f'\n=== GATE: {gate} (gap={gap:.4f}) ===')
if gate == 'NO-GO':
    print('STOP. Write up as a negative result. Do NOT continue to Phase 2.')

## Phase 2 (only if gate == GO)
Cells below are stubs. Do not run until Phase 1 gate passes.

In [ ]:
# Cell 7: Phase 2 stub — nonlinear ICL
assert verdict['gate'] == 'GO', 'Phase 1 gate is NO-GO. Stop here.'
# TODO: implement after Phase 1 GO